In [1]:
import pandas as pd
import glob
import os
import re

CSV_DIR = '../original_csv'

# Load Chase files (.CSV)
chase_files = glob.glob(os.path.join(CSV_DIR, '*.CSV'))

chase_frames = []
for f in chase_files:
    df = pd.read_csv(f)
    df = df[['Transaction Date', 'Description', 'Amount', 'Category']].copy()
    df.rename(columns={'Transaction Date': 'date', 'Category': 'category'}, inplace=True)
    df['account'] = 'chase'
    chase_frames.append(df)

chase = pd.concat(chase_frames, ignore_index=True)
chase['date'] = pd.to_datetime(chase['date'])

# Load Alliant files (.csv)
def parse_alliant_amount(val):
    val = str(val).strip()
    negative = val.startswith('(')
    val = re.sub(r'[\$,()\s]', '', val)
    amount = float(val)
    return -amount if negative else amount

alliant_files = glob.glob(os.path.join(CSV_DIR, '*.csv'))

alliant_frames = []
for f in alliant_files:
    df = pd.read_csv(f)
    df = df[['Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Date': 'date'}, inplace=True)
    df['Amount'] = df['Amount'].apply(parse_alliant_amount)
    df['account'] = 'alliant'
    alliant_frames.append(df)

alliant = pd.concat(alliant_frames, ignore_index=True)
alliant['date'] = pd.to_datetime(alliant['date'])

# Combine
combined = pd.concat([chase, alliant], ignore_index=True)
combined.rename(columns={'Amount': 'amount', 'Description': 'description'}, inplace=True)

combined['subcategory'] = pd.NA
combined['merchant'] = pd.NA
combined['ignore'] = False
combined['necessity'] = pd.NA

combined = combined[['date', 'description', 'amount', 'account', 'category', 'subcategory', 'merchant', 'ignore', 'necessity']]
combined.sort_values('date', inplace=True)
combined.reset_index(drop=True, inplace=True)

combined.to_csv('../all_transactions.csv')
print(f"Saved {len(combined)} rows to all_transactions.csv")

Saved 86 rows to all_transactions.csv
